# Cell-type spatial associations with the LNM map
Analyze physical CSV rows 2–25 (24 cell types), using row 1 as parcel headers and the last column as cell-type labels. Align parcels by their full surface-atlas names. Compute Pearson correlations, 5,000 bilateral spin nulls, and Benjamini–Hochberg FDR across cell types.

The LNM CSV uses global parcel IDs 1–400. The local surface atlases use IDs 1–200 in each hemisphere. Their ordering was audited against the corresponding Schaefer atlas. Expression headers use the same historical names as these surface atlases; do not match them directly to a newer naming version.

Missing expression values remain NaN; zeros remain valid observations. Spin correlations use pairwise finite observations and report their sample sizes.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
import nibabel as nib
from neuromaps.datasets import fetch_atlas
from brainspace.null_models import SpinPermutations
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

In [ ]:
work_dir = Path('xxxxxxxxx')
cell_csv = work_dir / '/cell_types/schaeffer_Jorstad_400_7Net_expr_mat_new_NormZscore0.3.csv'
lnm_csv = work_dir / 'xxxxxx/rumination_MDD_Schaefer400_values_GSR.csv'
out_dir = work_dir / 'xxxxxxx/cell_types'
fig_dir = work_dir / 'xxxxxxx/stable_260725/plots'
# Schaefer atlas
atlas_paths = {h: work_dir / f'atlas/fsaverage5_{h.lower()}_Schaefer2018_400Parcels_7Networks_order.label.gii' for h in ['LH', 'RH']}
result_csv = out_dir / 'LNM_cell_type_spin_results.csv'
fig_png = fig_dir / 'LNM_cell_type_spin_results.png'
n_perm = 5000
random_state = 42

In [ ]:
# Validate the actual label arrays and create explicit parcel-name mappings.
atlas_labels = {}
records = []
for hemi, path in atlas_paths.items():
    img = nib.load(path)
    arrays = [a for a in img.darrays if a.intent == 1002]
    assert len(arrays) == 1
    raw = arrays[0].data
    assert np.isfinite(raw).all() and np.equal(raw, np.rint(raw)).all()
    labels = raw.astype(int)
    assert labels.shape == (10242,)
    assert np.array_equal(np.unique(labels), np.arange(201))
    table = {int(x.key): getattr(x, 'label', '') for x in img.labeltable.labels}
    assert len(table) == len(img.labeltable.labels)
    assert set(table) == set(np.unique(labels))
    atlas_labels[hemi] = labels
    print(hemi, 'unique labels:', np.unique(labels), 'background:', int((labels == 0).sum()))
    for label_id in range(1, 201):
        name = table[label_id]
        assert name.startswith(f'7Networks_{hemi}_')
        records.append({'parcel_id': label_id + (200 if hemi == 'RH' else 0),
                        'parcel_name': name, 'hemi': hemi, 'surface_label_id': label_id})
parcels = pd.DataFrame(records).sort_values('parcel_id').reset_index(drop=True)
assert parcels.parcel_name.is_unique
assert np.array_equal(parcels.parcel_id, np.arange(1, 401))
parcel_name_to_index = dict(zip(parcels.parcel_name, range(400)))
print(parcels.to_string(index=False))

In [ ]:
# Physical rows 2–25 are the first 24 data rows after the header.
with cell_csv.open(newline='') as f:
    raw_rows = list(csv.reader(f))
assert len(raw_rows) >= 25
assert all(len(row) == 401 for row in raw_rows[:25])
header = raw_rows[0]
assert len(set(header[:400])) == 400, 'Duplicate parcel headers'
assert set(header[:400]) == set(parcels.parcel_name), 'Expression/atlas parcel names differ'
cell_types = pd.read_csv(cell_csv, nrows=24)
cell_labels = cell_types.iloc[:, -1].astype('string').str.strip()
assert cell_labels.notna().all() and cell_labels.ne('').all() and cell_labels.is_unique
expression = cell_types.iloc[:, :400].apply(pd.to_numeric, errors='raise')
# Reorder columns by parcel identity, never by their original positions alone.
expression = expression.loc[:, parcels.parcel_name]
assert expression.columns.tolist() == parcels.parcel_name.tolist()
expression.index = cell_labels.to_numpy()
assert expression.shape == (24, 400)
assert not np.isinf(expression.to_numpy(dtype=float)).any()

lnm_df = pd.read_csv(lnm_csv)
assert {'roi', 'hemi', 'score'} <= set(lnm_df)
assert lnm_df.roi.is_unique and set(lnm_df.roi) == set(range(1, 401))
lnm_df = lnm_df.set_index('roi').loc[parcels.parcel_id]
assert lnm_df.hemi.tolist() == ['left'] * 200 + ['right'] * 200
lnm_named = pd.Series(lnm_df.score.to_numpy(dtype=float), index=parcels.parcel_name)
assert lnm_named.index.tolist() == expression.columns.tolist()
lnm = lnm_named.to_numpy()
assert np.isfinite(lnm).all(), 'Inspect missing LNM values before generating spin nulls'
lnm_lh, lnm_rh = lnm[:200], lnm[200:]
print('Aligned expression shape:', expression.shape)
print('Finite parcels per cell type:', expression.notna().sum(axis=1).to_dict())

In [ ]:
# Use the identical parcel table for spin coordinates and expression/LNM values.
spheres = fetch_atlas('fsaverage', '10k')['sphere']
coords = {h: nib.load(path).darrays[0].data for h, path in zip(['LH', 'RH'], spheres)}
centroids = []
for row in parcels.itertuples(index=False):
    assert coords[row.hemi].shape == (len(atlas_labels[row.hemi]), 3)
    centroid = coords[row.hemi][atlas_labels[row.hemi] == row.surface_label_id].mean(axis=0)
    assert np.isfinite(centroid).all() and np.linalg.norm(centroid) > 0
    centroids.append(centroid / np.linalg.norm(centroid))
centroids = np.asarray(centroids)
coords_lh, coords_rh = centroids[:200], centroids[200:]
spin = SpinPermutations(n_rep=n_perm, random_state=random_state)
spin.fit(coords_lh, points_rh=coords_rh)
lnm_lh_spin, lnm_rh_spin = spin.randomize(lnm_lh, lnm_rh)
lnm_spin_all = np.concatenate([lnm_lh_spin, lnm_rh_spin], axis=1)
assert lnm_spin_all.shape == (n_perm, 400)

In [ ]:
def finite_pearson(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    n = int(valid.sum())
    if n < 3:
        return np.nan, n
    a, b = x[valid], y[valid]
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return np.nan, n
    return float(np.corrcoef(a, b)[0, 1]), n

# Reuse the same bilateral spin rotations for all 24 cell types.
results = []
for cell_label, row in expression.iterrows():
    values = row.to_numpy(dtype=float)
    r_obs, n_valid = finite_pearson(lnm, values)
    null_results = [finite_pearson(rotated, values) for rotated in lnm_spin_all]
    null_r = np.array([r for r, n in null_results])
    null_n = np.array([n for r, n in null_results])
    valid_null = np.isfinite(null_r)
    # Do not silently reduce the permutation denominator when a null is invalid.
    valid_test = np.isfinite(r_obs) and valid_null.all()
    p_spin = ((np.sum(np.abs(null_r) >= abs(r_obs)) + 1) / (n_perm + 1)
              if valid_test else np.nan)
    results.append({'cell_type': cell_label, 'r_obs': r_obs, 'p_spin': p_spin,
                    'n_parcels': n_valid, 'n_null_valid': int(valid_null.sum()),
                    'null_n_min': int(null_n.min()), 'null_n_max': int(null_n.max()),
                    'status': 'ok' if valid_test else 'invalid_correlation_or_null'})
results_df = pd.DataFrame(results)
results_df['p_fdr'] = np.nan
valid = results_df.p_spin.notna()
if valid.any():
    results_df.loc[valid, 'p_fdr'] = multipletests(results_df.loc[valid, 'p_spin'], method='fdr_bh')[1]
results_df = results_df.sort_values('r_obs', ascending=False)
out_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(result_csv, index=False)
parcels.to_csv(out_dir / 'cell_type_parcel_order.csv', index=False)
print(results_df.to_string(index=False))
print('Saved:', result_csv)

In [ ]:
# Plot correlations; one star indicates spin p < .05, two indicate FDR q < .05.
plot_df = results_df.sort_values('r_obs').dropna(subset=['r_obs'])
fig, ax = plt.subplots(figsize=(7, max(5, len(plot_df) * 0.28)))
y = np.arange(len(plot_df))
ax.barh(y, plot_df.r_obs, color=plt.get_cmap('tab20')(y % 20))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y, plot_df.cell_type)
for yi, row in zip(y, plot_df.itertuples(index=False)):
    star = '**' if row.p_fdr < .05 else ('*' if row.p_spin < .05 else '')
    ax.text(row.r_obs + (.01 if row.r_obs >= 0 else -.01), yi, star,
            va='center', ha='left' if row.r_obs >= 0 else 'right')
ax.set_xlabel('Spatial correlation with LNM (Pearson r)')
ax.set_ylabel('Cell type')
ax.spines[['top', 'right']].set_visible(False)
ax.margins(x=.15)
fig.tight_layout()
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_png, dpi=300, bbox_inches='tight')
fig.savefig(fig_png.with_suffix('.pdf'), bbox_inches='tight')
plt.show()